In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_NUM_CPU_DEVICES"] = "8"

import numpy as np

import jax
import jax.numpy as jnp
import jax.sharding as shd
import functools

In [2]:
P = shd.PartitionSpec

mesh = jax.make_mesh((2, 4), ('x', 'y'))
model_axes = ('x', 'y')
hidden_sharding = shd.NamedSharding(mesh, P(None, model_axes))
output_sharding = shd.NamedSharding(mesh, P(None, None))

d_model = 8192
batch = 1
ffw_mult = 4

cpu_device = jax.devices('cpu')[0]

cpu_x = jnp.zeros((batch, d_model), dtype=jnp.bfloat16, device=cpu_device)
cpu_w1 = jnp.zeros((ffw_mult * d_model, d_model), dtype=jnp.bfloat16, device=cpu_device)
cpu_w2 = jnp.zeros((d_model, ffw_mult * d_model), dtype=jnp.bfloat16, device=cpu_device)

def matmul(w1, w2, x):
  hidden = jnp.einsum('fw,bw->bf', w1, x, out_sharding=hidden_sharding)
  return jnp.einsum('wf,bf->bw', w2, hidden, out_sharding=output_sharding)

def make_sharding():
  return (shd.NamedSharding(mesh, P(model_axes, None)),
          shd.NamedSharding(mesh, P(None, model_axes)),
          shd.NamedSharding(mesh, P(None, None)))

w1_sharding, w2_sharding, x_sharding = make_sharding()

try:
  del x, w1, w2
except:
  pass

x, w1, w2 = jax.device_put(cpu_x, x_sharding), jax.device_put(cpu_w1, w1_sharding), jax.device_put(cpu_w2, w2_sharding)

compiled = jax.jit(matmul, in_shardings=make_sharding()).lower(w1, w2, x).compile()
result = compiled(w1, w2, x)


In [3]:
import jax

with jax.profiler.trace("./tensorboard"):
  y = compiled(w1, w2, x)
  y.block_until_ready()

In [4]:
%load_ext tensorboard

In [5]:
%tensorboard --logdir ./tensorboard